# TP — Améliorer l'utilisation d'un LLM

Durée indicative : 60 min.

**La tâche** : répondre aux questions d'une équipe sur un texte de loi récent, sous forme de fiche courte.

**La méthode** : la même tâche, les mêmes questions de test, une mesure identique, et quatre façons de s'y prendre :

| Étape | Technique | Ce qu'on attend |
|---|---|---|
| 0 | Prompt brut | Le point de départ |
| 1 | Prompt engineering | La forme devient stable |
| 2 | Mémoire | On ne répète plus les consignes |
| 3 | RAG | Les faits deviennent justes |

Chaque étape est mesurée par le même banc d'essai (`src/evaluation.py`) : format respecté, longueur, faits attendus présents, source juste, tokens consommés. Le tableau final compare les quatre.

### Comment compléter

Un seul bloc `TODO` par étape, délimité par `# ---- TODO` et `# ---- fin TODO`. Il s'agit chaque fois d'écrire **du texte** (un prompt, une consigne, un gabarit), jamais de code LangChain. Le code qui appelle le modèle est fourni.

In [ ]:
import sys
from pathlib import Path

# Le notebook s'exécute depuis la racine du dépôt, quel que soit le dossier d'ouverture
RACINE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path[:0] = [str(RACINE), str(RACINE / "01-rappels-llm")]

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

from common.llm import get_llm, get_provider, invoquer
from common.rag import formater_contexte, indexer_corpus, questions_test, rechercher
from src.evaluation import FORMAT_FICHE, Banc, evaluer, resume_mesure

CORPUS = "loi-repost"  # ou "orion" : même TP, autre corpus (voir data/corpus/*/README.md)

llm = get_llm()  # relit .env à chaque appel : modifier .env puis relancer cette cellule suffit
questions = questions_test(CORPUS)
banc = Banc(questions)

print("Fournisseur :", get_provider(), "·", llm.model if hasattr(llm, "model") else "")
print("Test :", invoquer(llm, "Réponds en un mot : ça marche ?").content)
print()
for q in questions:
    print("-", q["question"], "→ faits attendus :", q["faits_attendus"])

Les *faits attendus* sont des chaînes que la bonne réponse doit contenir ; la *source attendue*, ce que le champ SOURCE doit mentionner (le bon article). C'est grossier, mais mécanique : le score ne dépend pas de l'impression du lecteur.

### La seule chose à savoir sur LangChain

Un appel au modèle, c'est une **liste de messages** : des consignes permanentes (`SystemMessage`), ce que dit l'utilisateur (`HumanMessage`), ce qu'a répondu le modèle (`AIMessage`). Le modèle renvoie un `AIMessage` dont le texte est dans `.content`.

La fonction `appeler` ci-dessous construit cette liste. Tout le notebook passe par elle.

In [ ]:
def appeler(question: str, systeme: str | None = None, historique: list | None = None) -> AIMessage:
    """Un appel au modèle.

    question   : le message de l'utilisateur
    systeme    : consignes permanentes (persona, format, règles), optionnel
    historique : messages précédents de la conversation, optionnel
    """
    messages = []
    if systeme:
        messages.append(SystemMessage(systeme))
    messages.extend(historique or [])
    messages.append(HumanMessage(question))
    return invoquer(llm, messages)  # invoquer = llm.invoke + nouvel essai en cas d'erreur 429


reponse = appeler("Dis bonjour en une phrase.", systeme="Tu réponds toujours en alexandrins.")
print(reponse.content)
print(reponse.usage_metadata)

## Étape 0 — La tâche brute

La question, telle quelle, sans rien autour.

In [ ]:
def repondre_brut(question: str) -> AIMessage:
    return appeler(question)


mesures = banc.mesurer("0 - prompt brut", repondre_brut)

Trois choses à observer :

1. **Le fond** : la loi est postérieure à la date de coupure du modèle. Il ne peut pas la connaître. Il répond pourtant, avec des articles de code et des montants qui ont l'air plausibles. Comparer avec les faits attendus : ce sont d'autres textes, d'autres peines. Le modèle ne signale pas qu'il ne sait pas.
2. **La forme** : longueur, structure et ton varient d'une question à l'autre. Rien n'est exploitable par un programme.
3. **Le coût** : autour de 1 000 tokens de sortie par réponse. Si le compte affiche exactement 1024, c'est la borne `max_tokens` de `get_llm()` qui a coupé la réponse : sans elle, le modèle aurait continué.

Un piège de mesure : une réponse de 500 mots qui énumère toutes les peines imaginables finit par contenir « un an » ou « 3 750 » par hasard. La colonne *faits justes* peut donc être flatteuse ici. Elle ne se lit qu'avec les colonnes *mots* et *source juste* : des chiffres justes noyés dans du faux, sans référence, ne servent à rien.

Essai libre : reformuler une des questions à la main (préciser le contexte, demander une réponse courte, demander d'indiquer le degré de certitude) et relancer. Constater ce qui s'améliore, et ce qui ne tient pas d'une question à l'autre.

In [ ]:
questions[1]

In [ ]:
question_libre = "Quelle est la capitale de la france ?" # ou questions[1]["question"]  # à modifier
reponse = appeler(question_libre)
print(reponse.content)
print()
print(resume_mesure(evaluer(reponse, questions[1]))) # à modifier si nécessaire (1 ou 2 ou ..)

## Étape 1 — Prompt engineering

Le prompt système se construit par couches, dans l'ordre vu en cours :

1. **Persona** : qui parle, pour qui.
2. **Tâche** : ce qu'il faut produire.
3. **Contraintes et format** : la fiche imposée (`FORMAT_FICHE`), et la règle sur ce qu'on ne sait pas.
4. **Exemple** (one-shot) : une fiche complète sur une question sans rapport, pour montrer la forme attendue.
5. **Délimiteurs** : des titres `###` séparent les blocs, pour que le modèle ne mélange pas consignes et données.

Les couches 1, 2 et 5 sont fournies. Afficher `FORMAT_FICHE` avant de commencer :

In [ ]:
print(FORMAT_FICHE)

In [ ]:
PERSONA = """Tu es juriste dans une direction juridique. Tu réponds aux questions des équipes
opérationnelles, qui ne sont pas juristes : précis, sobre, sans jargon inutile."""

TACHE = """Pour chaque question, produis une fiche de réponse courte."""

# ---- TODO : les couches 3 et 4, en texte ----------------------------------------------
# CONTRAINTES : imposer le format (recopier FORMAT_FICHE, ou l'inclure avec {FORMAT_FICHE}),
#   puis les règles :
#   - pas de texte avant ni après la fiche, pas de mise en forme (pas de gras) ;
#   - si l'information n'est pas connue avec certitude : SOURCE = "non trouvee",
#     CONFIANCE = "basse", et aucun chiffre, date ou article inventé.
# EXEMPLE : une question sans rapport avec le corpus (ex. la durée légale du travail),
#   suivie d'une fiche complète au format exact.
CONTRAINTES = f"""..."""

EXEMPLE = """Question : ...

REPONSE : ...
DETAILS :
- ...
- ...
SOURCE : ...
CONFIANCE : ..."""
# ---- fin TODO ------------------------------------------------------------------------

SYSTEM = f"""{PERSONA}

{TACHE}

### Format de réponse
{CONTRAINTES}

### Exemple
{EXEMPLE}"""


def repondre_prompt(question: str) -> AIMessage:
    return appeler(question, systeme=SYSTEM)


mesures = banc.mesurer("1 - prompt travaille", repondre_prompt)

Vérification attendue : **format 3/3**, longueur divisée par cinq, faits justes proches de 0, **source juste 0/3**.

Regarder le champ SOURCE et le champ CONFIANCE. Le modèle cite des articles de code précis, avec « CONFIANCE : haute » : ce sont des références inventées, produites avec la même assurance qu'une vraie. La ligne `source inventee` du banc les repère parce que le bon article est connu. En production, personne ne le sait. La règle « non trouvee / basse » demande l'honnêteté ; le prompt engineering ne peut pas la garantir.

Si le format n'est pas respecté : vérifier que l'exemple suit exactement `FORMAT_FICHE`, champ par champ.

## Étape 2 — Mémoire

Une conversation à plusieurs tours : les consignes au premier tour, puis les questions. Un LLM n'a pas de mémoire : ce qui n'est pas renvoyé dans le contexte n'existe plus.

Trois stratégies, vues en cours :

- **aucune** : chaque tour est un appel isolé ;
- **buffer** : tout l'historique est renvoyé à chaque tour ;
- **résumé** : l'historique est compressé en un résumé, plus les deux derniers messages.

Ce qu'on mesure : le format tient-il d'un tour à l'autre, et combien de tokens partent à chaque appel.

La classe `Conversation` est fournie. Le `TODO` porte sur la consigne donnée au modèle pour résumer.

In [ ]:
class Conversation:
    """Historique en Python nu. Le module « mémoire » fera la même chose avec LangGraph."""

    def __init__(self, memoire: str = "aucune"):
        self.memoire = memoire
        self.historique: list = []  # alternance HumanMessage / AIMessage
        self.resume: str = ""

    def contexte(self) -> list:
        """Ce qui est renvoyé au modèle avant la nouvelle question, selon la stratégie."""
        if self.memoire == "aucune":
            return []
        if self.memoire == "buffer":
            return list(self.historique)
        if self.memoire == "resume":
            return [SystemMessage(f"Résumé de la conversation :\n{self.resume}"), *self.historique[-2:]]
        raise ValueError(self.memoire)

    def dire(self, texte: str) -> AIMessage:
        reponse = appeler(texte, historique=self.contexte())
        self.historique += [HumanMessage(texte), reponse]
        if self.memoire == "resume":
            self.resume = self.resumer()
        return reponse

    def resumer(self) -> str:
        # ---- TODO : la consigne de résumé, en texte -------------------------------------
        # Le résumé remplace tout l'historique au tour suivant. Dire au modèle ce qu'il doit
        # conserver en priorité (les consignes reçues : format, règles), puis ce qu'il peut
        # condenser (questions posées, faits établis), et la longueur visée.
        consigne = """..."""
        # ---- fin TODO -------------------------------------------------------------------
        return appeler("Rédige le résumé maintenant.", systeme=consigne, historique=self.historique).content


CONSIGNES_TOUR_1 = f"""{PERSONA}

{TACHE} Pour toute la suite de cette conversation, réponds exactement au format suivant :

{CONTRAINTES}

Réponds simplement "Compris" pour ce message."""


def conversation_mesuree(memoire: str) -> list[dict]:
    conv = Conversation(memoire)
    print(f"[{memoire}] tour 1 → {conv.dire(CONSIGNES_TOUR_1).content[:40]!r}")
    mesures = []
    for q in questions:
        reponse = conv.dire(q["question"])
        m = evaluer(reponse, q)
        mesures.append(m)
        print(f"[{memoire}] format {'OK' if m['format_ok'] else 'KO':2} · {m['tokens_entree']:5} tokens en entrée")
    return mesures


for memoire in ["aucune", "buffer", "resume"]:
    banc.enregistrer(f"2 - memoire {memoire}", conversation_mesuree(memoire))
    print()

Lecture attendue :

- **aucune** : le format donné au tour 1 est perdu dès le tour 2.
- **buffer** : le format tient, mais les tokens en entrée augmentent à chaque tour. Sur une longue conversation, le contexte finit par déborder, et chaque appel coûte de plus en plus.
- **résumé** : le format tient si le résumé a conservé les consignes. Sur trois tours, le résumé ne coûte pas moins que le buffer : le résumé lui-même pèse plusieurs centaines de tokens. L'écart apparaît quand la conversation s'allonge : le buffer croît sans fin, le résumé plafonne. Si le format est perdu, c'est que le résumé a laissé tomber une consigne : reprendre la consigne de résumé.

La mémoire n'a rien changé au fond : les faits sont toujours faux ou absents. Elle sert à ne pas répéter, pas à savoir.

Pour voir le résumé produit :

In [ ]:
conv = Conversation("resume")
conv.dire(CONSIGNES_TOUR_1)
conv.dire(questions[0]["question"])
print(conv.resume)

## Étape 3 — RAG

Indexer → retrouver → augmenter → générer. L'indexation et la recherche sont fournies par `common.rag` (Qdrant en mémoire, embeddings locaux, recherche hybride dense + BM25). Reste à écrire l'augmentation du prompt.

In [ ]:
n = indexer_corpus(CORPUS)
print(f"{n} passages indexés")

for p in rechercher(questions[1]["question"], k=3):
    ref = f"article {p['article']}" if p["article"] else p["section"]
    print(f"\n[{p['score']:.3f}] {p['source']}, {ref}\n{p['texte'][:300]}…")

Chaque passage porte sa référence (numéro d'article ou titre de section) : c'est ce qui permet au modèle de remplir le champ SOURCE avec autre chose que « non trouvee ».

`formater_contexte(passages)` met les passages en forme, numérotés, avec leur référence. Le message envoyé au modèle a trois blocs, séparés par des titres `###` : les passages, la question, la consigne.

In [ ]:
def repondre_rag(question: str, k: int = 4) -> AIMessage:
    passages = rechercher(question, k=k)
    contexte = formater_contexte(passages)
    # ---- TODO : le message augmenté, en texte -------------------------------------------
    # Trois blocs. Les variables `contexte` et `question` s'insèrent avec {contexte} et
    # {question}. La consigne dit : répondre uniquement à partir des passages ; reporter
    # dans SOURCE la référence du passage utilisé (fichier et article) ; si les passages ne
    # suffisent pas, appliquer la règle "non trouvee".
    message = f"""### Passages
...

### Question
...

### Consigne
..."""
    # ---- fin TODO ----------------------------------------------------------------------
    return appeler(message, systeme=SYSTEM)


mesures = banc.mesurer("3 - RAG", repondre_rag)

Vérification attendue : **faits justes 5/6 ou 6/6, source juste 3/3**, et des tokens en entrée multipliés par quatre. Le RAG corrige le fond, et c'est la seule étape qui le fait. Il coûte ce qu'il apporte. Les sources sont maintenant les vraies : le champ SOURCE pointe vers le fichier et l'article d'où vient la réponse.

Si un fait manque : afficher les passages retrouvés pour cette question. Le problème est soit dans la recherche (le bon passage n'est pas dans le top-k, essayer `k=6`), soit dans la génération (le passage est là, le modèle ne l'a pas utilisé).

## Le tableau final

In [ ]:
banc.tableau()

Une ligne par étape. À lire en colonnes :

| Colonne | Corrigée par | Prix |
|---|---|---|
| Format OK, Mots | le prompt engineering | quelques centaines de tokens de consignes |
| Format qui tient d'un tour à l'autre | la mémoire | des tokens à chaque tour ; le résumé plafonne, le buffer non |
| Faits justes, Source juste | le RAG | un contexte quatre fois plus gros |

Chaque technique corrige un défaut différent. Aucune ne remplace les autres.

### Ce qui reste

- Le modèle répond juste, dans la bonne forme, sans qu'on se répète. Mais il ne peut **rien faire** : pas de recherche, pas d'action, pas de vérification. C'est l'objet des outils et des agents.
- La mémoire est artisanale : une liste et un résumé. Le module mémoire la reprend avec un état typé, un checkpointer et des threads.
- Le RAG est naïf : il fait confiance au premier passage venu. Le module RAG réflexif ajoute un contrôle de pertinence et une boucle de correction.

## Bonus

**Sortie structurée.** Le format texte est fragile (un `**` de trop et le parseur échoue). Les fournisseurs savent produire directement un objet Pydantic validé.

In [ ]:
from pydantic import BaseModel, Field


class Fiche(BaseModel):
    reponse: str = Field(description="Une phrase, 40 mots maximum")
    details: list[str] = Field(description="2 ou 3 points")
    source: str = Field(description="Référence du passage utilisé, ou 'non trouvee'")
    confiance: str = Field(description="haute, moyenne ou basse")


llm_structure = llm.with_structured_output(Fiche)
passages = rechercher(questions[0]["question"], k=4)
fiche = invoquer(
    llm_structure,
    [SystemMessage(SYSTEM), HumanMessage(f"{formater_contexte(passages)}\n\nQuestion : {questions[0]['question']}")],
)
fiche

**Changer de corpus.** Remettre `CORPUS = "orion"` dans la cellule d'imports et tout relancer : même notebook, même banc, autre document. Les questions de test et les faits attendus viennent du `questions.json` du corpus.

**Changer de modèle.** `LLM_MODEL=mistralai:ministral-14b-latest` dans `.env`, puis relancer la cellule d'imports. Comparer les tableaux.